In [1]:
import re
import pandas as pd
from pathlib import Path

# ----------------------------------------
# 0) 경로 설정
# ----------------------------------------
PROJECT_ROOT = Path("/Users/gimgyumin/Desktop/Developer/commercial-area-analysis-ai")
RAW_DIR  = PROJECT_ROOT / "data" / "raw_data"
MAP_DIR  = PROJECT_ROOT / "data" / "category_maps"
OUT_DIR  = PROJECT_ROOT / "data" / "processed_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SALES_DIR     = RAW_DIR / "sales_info"
STORE_DIR     = RAW_DIR / "store_info"
POP_PATH      = RAW_DIR / "population_info" / "유동인구.csv"
BS_DIR        = RAW_DIR / "BS_info"
WORK_POP_PATH = RAW_DIR / "working-population_info" / "서울시_상권분석서비스(직장인구-행정동).csv"
RES_POP_DIR   = RAW_DIR / "residential-population_info"

SALES_MAP_PATH = MAP_DIR / "sales_category_map.csv"
STORE_MAP_PATH = MAP_DIR / "store_category_map.csv"

OUT_PATH = OUT_DIR / "final_dataset.csv"


# ----------------------------------------
# 1) 한글 CSV 안전 로더
# ----------------------------------------
def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols, low_memory=False)
        except Exception:
            pass
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols,
                               engine="python", on_bad_lines="skip")
        except Exception:
            pass
    raise RuntimeError(f"CSV 읽기 실패: {path}")


# ----------------------------------------
# 2) 카테고리 매핑 로드
# ----------------------------------------
sales_map_df = read_csv_kor(SALES_MAP_PATH)
store_map_df = read_csv_kor(STORE_MAP_PATH)

SALES_CATEGORY_MAP = dict(zip(sales_map_df["서비스_업종_코드_명"], sales_map_df["통합_카테고리"]))
STORE_CATEGORY_MAP = dict(zip(store_map_df["상권업종중분류명"].str.strip(), store_map_df["통합_카테고리"]))


# ----------------------------------------
# 3) 기존 원본 로드
# ----------------------------------------

# 매출: 연도별 파일 전체 합치기
sales_df = pd.concat(
    [read_csv_kor(f) for f in sorted(SALES_DIR.glob("매출_*.csv"))],
    ignore_index=True
)

# 유동인구: 단일 파일
pop_df = read_csv_kor(POP_PATH)

# 상가: 분기별 폴더 순회 → 폴더명으로 기준_년분기_코드 생성
store_parts = []
for folder in sorted(STORE_DIR.iterdir()):
    if not folder.is_dir():
        continue
    store_file = folder / "상가_서울.csv"
    if not store_file.exists():
        print(f"파일 없음 (건너뜀): {folder.name}")
        continue
    quarter_code = int(folder.name.replace("-", ""))  # "2020-1" → 20201
    df = read_csv_kor(store_file)
    df["기준_년분기_코드"] = quarter_code
    store_parts.append(df)

store_df = pd.concat(store_parts, ignore_index=True)

print(f"매출 행 수: {len(sales_df):,}")
print(f"유동인구 행 수: {len(pop_df):,}")
print(f"상가 행 수: {len(store_df):,}")
print(f"상가 분기: {sorted(store_df['기준_년분기_코드'].unique())}")


# ----------------------------------------
# 3-1) BS_info: 개업률 / 폐업률 / 프랜차이즈 점포수
# ----------------------------------------
bs_parts = []
for fn in sorted(BS_DIR.glob("*.csv")):
    bs_parts.append(read_csv_kor(fn))

bs_df = pd.concat(bs_parts, ignore_index=True)
bs_df = bs_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})
bs_df["행정동코드"] = pd.to_numeric(bs_df["행정동코드"], errors="coerce").astype("Int64")
bs_df["통합_카테고리"] = bs_df["서비스_업종_코드_명"].map(SALES_CATEGORY_MAP)
bs_df = bs_df.dropna(subset=["통합_카테고리"])

bs_agg = (
    bs_df.groupby(["기준_년분기_코드", "행정동코드", "통합_카테고리"], dropna=False)
    .agg(
        개업_율_평균=("개업_율", "mean"),
        폐업_률_평균=("폐업_률", "mean"),
        프랜차이즈_점포수=("프랜차이즈_점포_수", "sum"),
    )
    .reset_index()
)

print(f"\nBS_info 행 수: {len(bs_df):,}")
print(f"BS_info 분기 범위: {sorted(bs_df['기준_년분기_코드'].unique())[:4]} ... {sorted(bs_df['기준_년분기_코드'].unique())[-4:]}")


# ----------------------------------------
# 3-2) 직장인구 (전 연령대 + 성별 추가)
# ----------------------------------------
work_df = read_csv_kor(WORK_POP_PATH)
work_df = work_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})
work_df["행정동코드"] = pd.to_numeric(work_df["행정동코드"], errors="coerce").astype("Int64")

for col in ["총_직장_인구_수", "연령대_20_직장_인구_수", "연령대_30_직장_인구_수",
            "연령대_40_직장_인구_수", "연령대_50_직장_인구_수", "연령대_60_이상_직장_인구_수",
            "남성_직장_인구_수", "여성_직장_인구_수"]:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

work_agg = (
    work_df.groupby(["기준_년분기_코드", "행정동코드"], dropna=False)
    .agg(
        총_직장_인구_수=("총_직장_인구_수", "sum"),
        직장_20대_인구=("연령대_20_직장_인구_수", "sum"),
        직장_30대_인구=("연령대_30_직장_인구_수", "sum"),
        직장_40대_인구=("연령대_40_직장_인구_수", "sum"),
        직장_50대_인구=("연령대_50_직장_인구_수", "sum"),
        직장_60대이상_인구=("연령대_60_이상_직장_인구_수", "sum"),
        남성_직장_인구=("남성_직장_인구_수", "sum"),
        여성_직장_인구=("여성_직장_인구_수", "sum"),
    )
    .reset_index()
)

print(f"\n직장인구 행 수: {len(work_df):,}")
print(f"직장인구 분기 범위: {sorted(work_df['기준_년분기_코드'].unique())[:4]} ... {sorted(work_df['기준_년분기_코드'].unique())[-4:]}")


# ----------------------------------------
# 3-3) 주거인구 (wide → long)
# ----------------------------------------
def parse_quarter_str(q_str):
    """'2022. 1/4' → 20221"""
    m = re.match(r'(\d{4})\.\s*(\d)/4', str(q_str).strip())
    if m:
        return int(m.group(1)) * 10 + int(m.group(2))
    return None

res_parts = []
for fn in sorted(RES_POP_DIR.glob("*.csv")):
    df = read_csv_kor(fn)
    df = df[(df["구분별"] == "계") & (df["항목"] == "주민등록인구(동별)")]
    df = df[~df["동별"].isin(["합계"]) & ~df["동별"].str.endswith("구")]
    quarter_cols = [c for c in df.columns if re.match(r'\d{4}\.\s*\d/4', str(c).strip())]
    df = df[["동별"] + quarter_cols].copy()
    df = df.melt(id_vars="동별", var_name="분기_str", value_name="주거인구")
    df["기준_년분기_코드"] = df["분기_str"].map(parse_quarter_str)
    df = df.dropna(subset=["기준_년분기_코드"])
    df["기준_년분기_코드"] = df["기준_년분기_코드"].astype(int)
    df["주거인구"] = pd.to_numeric(df["주거인구"].astype(str).str.replace(",", ""), errors="coerce")
    df = df.rename(columns={"동별": "행정동명"})
    res_parts.append(df[["기준_년분기_코드", "행정동명", "주거인구"]])

res_df = pd.concat(res_parts, ignore_index=True)
res_df = res_df.groupby(["기준_년분기_코드", "행정동명"])["주거인구"].sum().reset_index()

print(f"\n주거인구 행 수: {len(res_df):,}")
print(f"주거인구 분기 범위: {sorted(res_df['기준_년분기_코드'].unique())[:4]} ... {sorted(res_df['기준_년분기_코드'].unique())[-4:]}")


# ----------------------------------------
# 4) 컬럼명 통일
# ----------------------------------------
sales_df = sales_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})
pop_df   = pop_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})

for df in (sales_df, pop_df, store_df):
    df["행정동코드"] = pd.to_numeric(df["행정동코드"], errors="coerce").astype("Int64")


# ----------------------------------------
# 5) 매출 집계 (분기 × 행정동 × 업종) - 전 연령대 + 성별 + 시간대 + 주중/주말 추가
# ----------------------------------------
sales_df["통합_카테고리"] = sales_df["서비스_업종_코드_명"].map(SALES_CATEGORY_MAP)
sales_df = sales_df.dropna(subset=["통합_카테고리"]).copy()

sales_agg = (
    sales_df.groupby(["기준_년분기_코드", "행정동코드", "통합_카테고리"], dropna=False)
    .agg(
        당월매출합=("당월_매출_금액", "sum"),
        당월매출건수=("당월_매출_건수", "sum"),          # 객단가 계산용
        매출_10대합=("연령대_10_매출_금액", "sum"),
        매출_20대합=("연령대_20_매출_금액", "sum"),
        매출_30대합=("연령대_30_매출_금액", "sum"),
        매출_40대합=("연령대_40_매출_금액", "sum"),
        매출_50대합=("연령대_50_매출_금액", "sum"),
        매출_60대이상합=("연령대_60_이상_매출_금액", "sum"),
        남성매출합=("남성_매출_금액", "sum"),
        여성매출합=("여성_매출_금액", "sum"),
        주중매출합=("주중_매출_금액", "sum"),
        주말매출합=("주말_매출_금액", "sum"),
        시간대_00_06_매출=("시간대_00~06_매출_금액", "sum"),
        시간대_06_11_매출=("시간대_06~11_매출_금액", "sum"),
        시간대_11_14_매출=("시간대_11~14_매출_금액", "sum"),
        시간대_14_17_매출=("시간대_14~17_매출_금액", "sum"),
        시간대_17_21_매출=("시간대_17~21_매출_금액", "sum"),
        시간대_21_24_매출=("시간대_21~24_매출_금액", "sum"),
    )
    .reset_index()
)


# ----------------------------------------
# 6) 상가 집계 (분기 × 행정동 × 업종 → 점포수)
# ----------------------------------------
store_df["상권업종중분류명"] = store_df["상권업종중분류명"].str.strip()
store_df["상권업종소분류명"] = store_df["상권업종소분류명"].str.strip()
store_df["통합_카테고리"] = store_df["상권업종중분류명"].map(STORE_CATEGORY_MAP)


# 행정동 재편 대응: 2025-3부터 용신동 일부가 신설동/용두동으로 분리됨
# 매출/인구 등 다른 데이터는 여전히 용신동(11230536) 코드를 사용하므로 통일
DONG_REMAP_CODE = {11230515: 11230536, 11230533: 11230536}
DONG_REMAP_NAME = {11230515: "용신동",  11230533: "용신동"}
remap_mask = store_df["행정동코드"].isin(DONG_REMAP_CODE)
store_df.loc[remap_mask, "행정동명"]   = store_df.loc[remap_mask, "행정동코드"].map(DONG_REMAP_NAME)
store_df.loc[remap_mask, "행정동코드"] = store_df.loc[remap_mask, "행정동코드"].map(DONG_REMAP_CODE)

_fastfood = {"치킨", "피자", "버거", "토스트/샌드위치/샐러드"}
_mask = (store_df["상권업종중분류명"] == "기타 간이") & (store_df["상권업종소분류명"].isin(_fastfood))
store_df.loc[_mask, "통합_카테고리"] = "패스트푸드/치킨"

store_df = store_df.dropna(subset=["통합_카테고리"]).copy()

store_agg = (
    store_df.groupby(["기준_년분기_코드", "행정동코드", "통합_카테고리"], dropna=False)
    .size()
    .reset_index(name="점포수")
)

dong_name = store_df[["행정동코드", "행정동명"]].drop_duplicates()


# ----------------------------------------
# 7) 유동인구 집계 (분기 × 행정동) - 전 연령대 + 성별 + 시간대 추가
# ----------------------------------------
pop_agg = (
    pop_df.groupby(["기준_년분기_코드", "행정동코드"], dropna=False)
    .agg(
        총유동인구=("총_유동인구_수", "sum"),
        유동_10대=("연령대_10_유동인구_수", "sum"),
        유동_20대=("연령대_20_유동인구_수", "sum"),
        유동_30대=("연령대_30_유동인구_수", "sum"),
        유동_40대=("연령대_40_유동인구_수", "sum"),
        유동_50대=("연령대_50_유동인구_수", "sum"),
        유동_60대이상=("연령대_60_이상_유동인구_수", "sum"),
        남성유동=("남성_유동인구_수", "sum"),
        여성유동=("여성_유동인구_수", "sum"),
        시간대_00_06_유동=("시간대_00_06_유동인구_수", "sum"),
        시간대_06_11_유동=("시간대_06_11_유동인구_수", "sum"),
        시간대_11_14_유동=("시간대_11_14_유동인구_수", "sum"),
        시간대_14_17_유동=("시간대_14_17_유동인구_수", "sum"),
        시간대_17_21_유동=("시간대_17_21_유동인구_수", "sum"),
        시간대_21_24_유동=("시간대_21_24_유동인구_수", "sum"),
    )
    .reset_index()
)


# ----------------------------------------
# 8) 병합
# ----------------------------------------
merged = pd.merge(
    sales_agg,
    store_agg,
    on=["기준_년분기_코드", "행정동코드", "통합_카테고리"],
    how="left"
)

merged = pd.merge(
    merged,
    pop_agg,
    on=["기준_년분기_코드", "행정동코드"],
    how="left"
)

merged = pd.merge(
    merged,
    bs_agg,
    on=["기준_년분기_코드", "행정동코드", "통합_카테고리"],
    how="left"
)

merged = pd.merge(
    merged,
    work_agg,
    on=["기준_년분기_코드", "행정동코드"],
    how="left"
)

merged = pd.merge(
    merged,
    dong_name,
    on="행정동코드",
    how="left"
)

merged = pd.merge(
    merged,
    res_df,
    on=["기준_년분기_코드", "행정동명"],
    how="left"
)


# ----------------------------------------
# 9) 행정동 전체 집계 파생 컬럼
# ----------------------------------------
dong_total = (
    merged.groupby(["기준_년분기_코드", "행정동코드"], dropna=False)
    .agg(
        행정동_전체매출=("당월매출합", "sum"),
        행정동_전체점포수=("점포수", "sum"),
    )
    .reset_index()
)

merged = pd.merge(merged, dong_total, on=["기준_년분기_코드", "행정동코드"], how="left")


# ----------------------------------------
# 10) 파생 feature 계산
# ----------------------------------------
merged["점포수"] = merged["점포수"].fillna(0)

# 기존 피처
merged["업종_점포당매출"]  = merged["당월매출합"] / merged["점포수"].replace(0, pd.NA)
merged["업종_매출점유율"]  = merged["당월매출합"] / merged["행정동_전체매출"].replace(0, pd.NA)
merged["업종_포화도"]      = merged["점포수"] / merged["행정동_전체점포수"].replace(0, pd.NA)
merged["경쟁강도"]         = merged["점포수"]
merged["유동대비매출"]     = merged["당월매출합"] / merged["총유동인구"].replace(0, pd.NA)
merged["점포대비유동"]     = merged["총유동인구"] / merged["점포수"].replace(0, pd.NA)

# --- 가격대 지표 ---
merged["객단가"] = merged["당월매출합"] / merged["당월매출건수"].replace(0, pd.NA)

# --- 연령대별 매출 비율 ---
merged["매출_10대비율"]     = merged["매출_10대합"]    / merged["당월매출합"].replace(0, pd.NA)
merged["매출_20대비율"]     = merged["매출_20대합"]    / merged["당월매출합"].replace(0, pd.NA)
merged["매출_30대비율"]     = merged["매출_30대합"]    / merged["당월매출합"].replace(0, pd.NA)
merged["매출_40대비율"]     = merged["매출_40대합"]    / merged["당월매출합"].replace(0, pd.NA)
merged["매출_50대비율"]     = merged["매출_50대합"]    / merged["당월매출합"].replace(0, pd.NA)
merged["매출_60대이상비율"] = merged["매출_60대이상합"] / merged["당월매출합"].replace(0, pd.NA)

# --- 성별 매출 비율 ---
merged["매출_남성비율"] = merged["남성매출합"] / merged["당월매출합"].replace(0, pd.NA)
merged["매출_여성비율"] = merged["여성매출합"] / merged["당월매출합"].replace(0, pd.NA)

# --- 주중/주말 매출 비율 ---
merged["매출_주말비율"] = merged["주말매출합"] / merged["당월매출합"].replace(0, pd.NA)

# --- 시간대별 매출 비율 (타겟층 행동 패턴) ---
merged["매출_점심비율"] = merged["시간대_11_14_매출"] / merged["당월매출합"].replace(0, pd.NA)
merged["매출_저녁비율"] = merged["시간대_17_21_매출"] / merged["당월매출합"].replace(0, pd.NA)
merged["매출_심야비율"] = merged["시간대_21_24_매출"] / merged["당월매출합"].replace(0, pd.NA)

# --- 연령대별 유동인구 비율 ---
merged["유동_20대비율"]    = merged["유동_20대"]    / merged["총유동인구"].replace(0, pd.NA)
merged["유동_30대비율"]    = merged["유동_30대"]    / merged["총유동인구"].replace(0, pd.NA)
merged["유동_40대비율"]    = merged["유동_40대"]    / merged["총유동인구"].replace(0, pd.NA)
merged["유동_50대비율"]    = merged["유동_50대"]    / merged["총유동인구"].replace(0, pd.NA)

# --- 성별 유동인구 비율 ---
merged["유동_여성비율"] = merged["여성유동"] / merged["총유동인구"].replace(0, pd.NA)

# --- 직장인구 연령대/성별 비율 ---
merged["직장_20대_비율"]    = merged["직장_20대_인구"]    / merged["총_직장_인구_수"].replace(0, pd.NA)
merged["직장_30대_비율"]    = merged["직장_30대_인구"]    / merged["총_직장_인구_수"].replace(0, pd.NA)
merged["직장_40대_비율"]    = merged["직장_40대_인구"]    / merged["총_직장_인구_수"].replace(0, pd.NA)
merged["직장_여성비율"]     = merged["여성_직장_인구"]    / merged["총_직장_인구_수"].replace(0, pd.NA)

# --- 기존 MZ 지표 ---
merged["MZ_차이"] = merged["매출_20대비율"] - merged["유동_20대비율"]

merged = merged.rename(columns={"통합_카테고리": "통합카테고리"})


# ----------------------------------------
# 11) 저장
# ----------------------------------------
col_order = [
    # 키
    "기준_년분기_코드", "행정동코드", "통합카테고리", "행정동명",

    # 매출 원본
    "당월매출합", "당월매출건수",
    "매출_10대합", "매출_20대합", "매출_30대합",
    "매출_40대합", "매출_50대합", "매출_60대이상합",
    "남성매출합", "여성매출합",
    "주중매출합", "주말매출합",
    "시간대_00_06_매출", "시간대_06_11_매출", "시간대_11_14_매출",
    "시간대_14_17_매출", "시간대_17_21_매출", "시간대_21_24_매출",

    # 유동인구 원본
    "총유동인구",
    "유동_10대", "유동_20대", "유동_30대",
    "유동_40대", "유동_50대", "유동_60대이상",
    "남성유동", "여성유동",
    "시간대_00_06_유동", "시간대_06_11_유동", "시간대_11_14_유동",
    "시간대_14_17_유동", "시간대_17_21_유동", "시간대_21_24_유동",

    # 직장/주거인구
    "총_직장_인구_수",
    "직장_20대_인구", "직장_30대_인구", "직장_40대_인구", "직장_50대_인구", "직장_60대이상_인구",
    "남성_직장_인구", "여성_직장_인구",
    "주거인구",

    # 점포 정보
    "점포수", "행정동_전체매출", "행정동_전체점포수",
    "개업_율_평균", "폐업_률_평균", "프랜차이즈_점포수",

    # 파생 피처 - 시장 구조
    "업종_점포당매출", "업종_매출점유율", "업종_포화도",
    "경쟁강도", "유동대비매출", "점포대비유동",

    # 파생 피처 - 가격대
    "객단가",

    # 파생 피처 - 연령대 비율 (매출)
    "매출_10대비율", "매출_20대비율", "매출_30대비율",
    "매출_40대비율", "매출_50대비율", "매출_60대이상비율",

    # 파생 피처 - 성별 비율
    "매출_남성비율", "매출_여성비율",

    # 파생 피처 - 시간/요일 패턴
    "매출_주말비율", "매출_점심비율", "매출_저녁비율", "매출_심야비율",

    # 파생 피처 - 유동인구 비율
    "유동_20대비율", "유동_30대비율", "유동_40대비율", "유동_50대비율", "유동_여성비율",

    # 파생 피처 - 직장인구 비율
    "직장_20대_비율", "직장_30대_비율", "직장_40대_비율", "직장_여성비율",

    # 기존 복합 지표
    "MZ_차이",
]
merged = merged[col_order].dropna(subset=["통합카테고리"])
merged.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"\n저장 완료: {OUT_PATH}")
print(f"행 수: {len(merged):,}")
print(f"컬럼 수: {merged.shape[1]}")
print(f"분기 범위: {sorted(merged['기준_년분기_코드'].unique())}")
print(merged.head(3))

매출 행 수: 458,616
유동인구 행 수: 11,475
상가 행 수: 12,180,927
상가 분기: [np.int64(20201), np.int64(20202), np.int64(20203), np.int64(20204), np.int64(20211), np.int64(20212), np.int64(20213), np.int64(20214), np.int64(20221), np.int64(20222), np.int64(20223), np.int64(20224), np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254)]



BS_info 행 수: 682,931
BS_info 분기 범위: [np.int64(20191), np.int64(20192), np.int64(20193), np.int64(20194)] ... [np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253)]



직장인구 행 수: 11,178
직장인구 분기 범위: [np.int64(20191), np.int64(20192), np.int64(20193), np.int64(20194)] ... [np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253)]



주거인구 행 수: 11,960
주거인구 분기 범위: [np.int64(20191), np.int64(20192), np.int64(20193), np.int64(20194)] ... [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254)]



저장 완료: /Users/gimgyumin/Desktop/Developer/commercial-area-analysis-ai/data/processed_data/final_dataset.csv
행 수: 236,650
컬럼 수: 81
분기 범위: [np.int64(20191), np.int64(20192), np.int64(20193), np.int64(20194), np.int64(20201), np.int64(20202), np.int64(20203), np.int64(20204), np.int64(20211), np.int64(20212), np.int64(20213), np.int64(20214), np.int64(20221), np.int64(20222), np.int64(20223), np.int64(20224), np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253)]
   기준_년분기_코드     행정동코드  통합카테고리   행정동명       당월매출합  당월매출건수   매출_10대합  \
0      20191  11110515     미용실  청운효자동   179863795    5853   2057206   
1      20191  11110515   분식/간식  청운효자동  1105094602   90326  10759891   
2      20191  11110515  뷰티/화장품  청운효자동    10000000       5         0   

     매출_20대합    매출_30대합    매출_40대합  ...  유동_20대비율  유동_30대비율  유동_40대비율  \
0   23038977   29915560   57393052  ...   0.